In [0]:
# Widget for environment control
dbutils.widgets.text("catalog", "anurag_dev")
dbutils.widgets.text("schema", "bronze")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# Use the volume path
volume_path = f"/Volumes/{catalog}/{schema}/raw"

tables = ["customers", "orders", "products", "order_items"]

for table in tables:
    df = spark.read.option("header", True).option("inferSchema", True) \
        .csv(f"{volume_path}/{table}.csv")
    
    # Add ingestion metadata (Task 1 requirement)
    from pyspark.sql.functions import current_timestamp, lit, input_file_name
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_file_name", lit(f"{table}.csv"))
    
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog}.{schema}.{table}")
    
    print(f"✅ Loaded {table} — {df.count()} rows")